# Connectors and Integration
**Modules 5-6 · Intermediate** · part 3 of 8 of *Building Automated Data Pipelines with Spark, dbt and Airflow* (UK · USA · Mauritius job-market aligned)

Each notebook is **self-contained**: run top to bottom. Needs Python 3.9+ and Java 8/11/17 for PySpark (not needed for 01, 05, 08).

**Series:**
- `01_DFD_Data_Flow_Diagrams.ipynb`: Modules 1-2 · Beginner
- `02_Modular_Pipeline.ipynb`: Modules 3-4 · Beginner to Intermediate
- `03_Connectors_and_Integration.ipynb`: Modules 5-6 · Intermediate
- `04_SCD2_Spark_and_dbt.ipynb`: Modules 7-8 · Intermediate to Advanced
- `05_Airflow_Workflow_Design.ipynb`: Modules 9-10 · Advanced
- `06_Capstone_End_to_End.ipynb`: Module 11 · Advanced
- `07_Advanced_Performance_Testing_CICD.ipynb`: Beyond the course · Advanced
- `08_Career_UK_USA_Mauritius.ipynb`: Job-market alignment · All levels

## Setup

In [ ]:
# In Colab / fresh env, uncomment:
# !pip install -q pyspark==3.5.3 pandas numpy matplotlib pyyaml

import os, sys, json, time, shutil, logging, sqlite3, ast
from pathlib import Path
from dataclasses import dataclass, field
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml
from pyspark.sql import SparkSession, DataFrame, functions as F, Window

spark = (SparkSession.builder.master("local[*]").appName("pipelines-lab")
         .config("spark.sql.shuffle.partitions", "8")
         .config("spark.sql.adaptive.enabled", "true")
         .config("spark.ui.showConsoleProgress", "false")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

BASE = Path("pipeline_lab"); BASE.mkdir(exist_ok=True)
DATA = BASE / "data"; DATA.mkdir(exist_ok=True)
WH   = BASE / "warehouse"
print("Spark", spark.version, "| workspace:", BASE.resolve())

## Lab data (simulated source systems)
We simulate three real-world sources, with *deliberately dirty* data (duplicates, bad amounts, nulls, mixed-case currency, late-arriving customer) so you practise real problems.

In [ ]:
rng = np.random.default_rng(42)

# --- customers: day 1 vs day 2 (changes + new customer) ---
cust_day1 = pd.DataFrame({
    "customer_id": [1, 2, 3, 4, 5, 6],
    "name":    ["Asha Patel", "Liam Smith", "Chloe Dubois", "Raj Ramgoolam", "Emma Jones", "Noah Brown"],
    "email":   [f"user{i}@example.com" for i in range(1, 7)],
    "city":    ["London", "Manchester", "Port Louis", "Curepipe", "New York", "Austin"],
    "segment": ["gold", "silver", "silver", "bronze", "gold", "bronze"],
})
cust_day2 = cust_day1.copy()
cust_day2.loc[cust_day2.customer_id == 2, "city"] = "Edinburgh"      # customer moved
cust_day2.loc[cust_day2.customer_id == 4, "segment"] = "silver"      # upgraded
cust_day2 = pd.concat([cust_day2, pd.DataFrame([{"customer_id": 7, "name": "Priya Nair", "email": "user7@example.com",
                                                 "city": "Ebene", "segment": "bronze"}])], ignore_index=True)

def simulate_source(day: int):
    con = sqlite3.connect(DATA / "crm.db")
    (cust_day1 if day == 1 else cust_day2).to_sql("customers", con, if_exists="replace", index=False)
    con.close()

# --- orders CSV (dirty on purpose) ---
n = 40
orders = pd.DataFrame({
    "order_id": np.arange(1000, 1000 + n),
    "customer_id": rng.integers(1, 8, n),
    "order_date": (pd.Timestamp("2026-09-01") + pd.to_timedelta(rng.integers(0, 28, n), unit="D")).strftime("%Y-%m-%d"),
    "amount": rng.uniform(5, 300, n).round(2),
    "currency": rng.choice(["GBP", "USD", "MUR"], n),
})
dirty = pd.concat([orders.iloc[[0, 1]],                                                       # duplicates
                   pd.DataFrame([{"order_id": 2001, "customer_id": 3, "order_date": "2026-09-10", "amount": -5.0, "currency": "GBP"},   # negative amount
                                 {"order_id": None, "customer_id": 2, "order_date": "2026-09-11", "amount": 20.0, "currency": "USD"},  # null key
                                 {"order_id": 2003, "customer_id": 1, "order_date": "2026-09-12", "amount": 99.0, "currency": " gbp "}])])  # messy currency
orders = pd.concat([orders, dirty], ignore_index=True)
orders.to_csv(DATA / "orders.csv", index=False)

# --- web events JSON lines ---
events = pd.DataFrame({
    "customer_id": rng.integers(1, 8, 60),
    "event_type": rng.choice(["view", "click", "add_to_cart", "purchase"], 60),
    "ts": (pd.Timestamp("2026-09-01") + pd.to_timedelta(rng.integers(0, 28 * 24 * 3600, 60), unit="s")).strftime("%Y-%m-%dT%H:%M:%S"),
})
events.to_json(DATA / "events.jsonl", orient="records", lines=True)

simulate_source(1)
print("orders rows:", len(orders), "| events:", len(events), "| customers (day1):", len(cust_day1))
orders.tail(6)

## Prerequisites from earlier notebooks (run once; skim if you already did them)

In [ ]:
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
log = logging.getLogger("pipeline")

def timed(fn):
    def wrapper(*a, **kw):
        t = time.time(); out = fn(*a, **kw)
        log.info("step=%s duration=%.2fs", fn.__name__, time.time() - t)
        return out
    wrapper.__name__ = fn.__name__
    return wrapper

@timed
def clean_orders(df: DataFrame):
    # Standardise types and split into (valid, quarantined)
    typed = (df.withColumn("order_id", F.col("order_id").cast("long"))
               .withColumn("customer_id", F.col("customer_id").cast("int"))
               .withColumn("order_date", F.to_date("order_date"))
               .withColumn("amount", F.col("amount").cast("double"))
               .withColumn("currency", F.upper(F.trim("currency"))))
    is_valid = F.coalesce(F.col("order_id").isNotNull() & (F.col("amount") > 0) & F.col("order_date").isNotNull(), F.lit(False))
    good = typed.filter(is_valid).dropDuplicates(["order_id"])
    bad = typed.filter(~is_valid).withColumn("reject_reason", F.lit("null key / non-positive amount / bad date"))
    return good, bad

raw_orders = spark.read.option("header", True).csv(str(DATA / "orders.csv"))
orders_ok, orders_bad = clean_orders(raw_orders)
print("raw:", raw_orders.count(), "| valid:", orders_ok.count(), "| quarantined:", orders_bad.count())
orders_bad.show(truncate=False)

In [ ]:
# Data-quality framework: (name, check, severity). 'error' blocks the pipeline, 'warn' only logs.
ORDER_CHECKS = [
    ("order_id is unique",       lambda df: df.count() == df.select("order_id").distinct().count(), "error"),
    ("order_id not null",        lambda df: df.filter("order_id IS NULL").count() == 0,              "error"),
    ("amount > 0",               lambda df: df.filter("amount <= 0").count() == 0,                   "error"),
    ("currency in allowed set",  lambda df: df.filter(~F.col("currency").isin("GBP", "USD", "MUR")).count() == 0, "error"),
    ("row count > 0",            lambda df: df.count() > 0,                                          "error"),
]

def run_dq(df, checks):
    results = []
    for name, fn, severity in checks:
        passed = bool(fn(df))
        results.append({"check": name, "passed": passed, "severity": severity})
        (log.info if passed else log.warning)("DQ %-28s %s", name, "PASS" if passed else "FAIL")
    failures = [r for r in results if not r["passed"] and r["severity"] == "error"]
    if failures:
        raise ValueError(f"Data quality gate failed: {failures}")
    return results

dq_results = run_dq(orders_ok, ORDER_CHECKS)
pd.DataFrame(dq_results)

# PART 3: Connectors and unified data integration (Modules 5–6) · Intermediate

A **connector** hides *how* we read a system behind a common `read()` interface. Add a new source by adding config, not by rewriting the pipeline. In production you would use JDBC (Postgres/SQL Server), S3/ADLS/GCS, REST APIs, Kafka or Fivetran/Airbyte; here we use SQLite, CSV and JSON so everything runs offline.

In [ ]:
from abc import ABC, abstractmethod

class Connector(ABC):
    def __init__(self, cfg): self.cfg = cfg
    @abstractmethod
    def read(self, spark) -> DataFrame: ...

class CSVConnector(Connector):
    def read(self, spark):
        return spark.read.options(**{"header": "true", **self.cfg.get("options", {})}).csv(self.cfg["path"])

class JSONConnector(Connector):
    def read(self, spark):
        return spark.read.json(self.cfg["path"])

class SQLiteConnector(Connector):          # stand-in for JDBC: spark.read.format("jdbc").option("url", ...).load()
    def read(self, spark):
        con = sqlite3.connect(self.cfg["path"])
        pdf = pd.read_sql_query(f"SELECT * FROM {self.cfg['table']}", con); con.close()
        return spark.createDataFrame(pdf)

REGISTRY = {"csv": CSVConnector, "json": JSONConnector, "sqlite": SQLiteConnector}
def build_connector(cfg): return REGISTRY[cfg["type"]](cfg)

CONFIG_YAML = f'''
sources:
  crm_customers:
    type: sqlite
    path: {DATA / "crm.db"}
    table: customers
  orders:
    type: csv
    path: {DATA / "orders.csv"}
  web_events:
    type: json
    path: {DATA / "events.jsonl"}
'''
CFG = yaml.safe_load(CONFIG_YAML)
raw = {name: build_connector(c).read(spark) for name, c in CFG["sources"].items()}
for k, v in raw.items():
    print(f"--- {k}: {v.count()} rows"); v.printSchema()

### Unified integration → Customer-360
Join heterogeneous sources into one business entity. Notice the **multi-currency trap**: never `SUM(amount)` across currencies. We convert with an (illustrative) FX table using a **broadcast join**.

In [ ]:
FX = spark.createDataFrame([("GBP", 1.0), ("USD", 0.78), ("MUR", 0.017)], ["currency", "to_gbp"])   # illustrative rates only

def build_customer_360(customers_current: DataFrame, orders_clean: DataFrame, events_raw: DataFrame) -> DataFrame:
    o = (orders_clean.join(F.broadcast(FX), "currency", "left")
                     .groupBy("customer_id")
                     .agg(F.count("*").alias("orders"),
                          F.round(F.sum(F.col("amount") * F.col("to_gbp")), 2).alias("revenue_gbp"),
                          F.max("order_date").alias("last_order_date")))
    e = events_raw.groupBy("customer_id").agg(F.count("*").alias("web_events"),
                                              F.sum((F.col("event_type") == "purchase").cast("int")).alias("web_purchases"))
    return (customers_current.join(o, "customer_id", "left").join(e, "customer_id", "left")
            .fillna({"orders": 0, "revenue_gbp": 0.0, "web_events": 0, "web_purchases": 0}))

c360 = build_customer_360(raw["crm_customers"], orders_ok, raw["web_events"])
c360.orderBy("customer_id").show()
# Orphan check: orders whose customer does not exist yet (late-arriving dimension)
print("orphan orders:", orders_ok.join(raw["crm_customers"], "customer_id", "left_anti").count())

**Teaching point: late-arriving dimensions.** Customer 7 has orders but is not in the CRM yet (day 1). Real pipelines either keep a placeholder "unknown" member or park orphans until the dimension arrives. Say this in interviews.

**Exercise 3.1:** add a `REST` connector skeleton (use `requests`, pagination, retries with backoff). **3.2:** add a `parquet` connector.

In [ ]:
spark.stop()